# Phase 0 dry-run on Kaggle — harness validation

Confirms the environment + the session-safe trainer (checkpoint / resume / determinism)
work on Kaggle hardware **before** any GPU training in Phase 1b.

**Settings (right panel):**
- Accelerator: **GPU T4 x2** or **P100** (not required for Phase 0, but use the same
  instance you'll train on).
- Internet: **On** (needed to clone + pip install).

Nothing here downloads models or datasets — it's a pure harness check.

## 1. Get the code (clone or update)

Cloned into a **colon-free** path (`/kaggle/working/latent-reasoning`). Re-running the
cell pulls the latest `main` instead of failing. The repo must be **public** (or add a
token to the URL for a private repo).

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
REPO_DIR = "/kaggle/working/latent-reasoning"  # no colon in the path

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 2. Install dependencies

Kaggle's base image already ships CUDA + torch, so this mostly resolves to no-ops plus
the light packages (`pyyaml`, etc.).

In [ ]:
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Run the test suite (Phase 0 + Phase 1a gates)

All CPU, no downloads. Expect every test to pass.

In [ ]:
!python -m pytest -q

## 4. Run the session-safe trainer

Drives the synthetic task to completion, checkpointing on cadence into
`outputs/phase0/checkpoints/`.

In [ ]:
!python -m src.train.kaggle_run --config configs/phase0.yaml

## 5. Prove cross-invocation resume

This is the property that lets Phase 1b survive Kaggle's ~9–12h session cap. We train
part-way, then re-run to a higher step count against the **same** `output_dir`: the second
run prints `[resume] continuing from step N` and finishes without redoing work.

(We use a fresh `output_dir` and step overrides so the demo is self-contained.)

In [ ]:
!rm -rf outputs/resume_demo
print("=== run A: steps 0 -> 20 ===")
!python -m src.train.kaggle_run --config configs/phase0.yaml \
    --set output_dir=outputs/resume_demo train.total_steps=20
print("\n=== run B: resume 20 -> 40 (same output_dir) ===")
!python -m src.train.kaggle_run --config configs/phase0.yaml \
    --set output_dir=outputs/resume_demo train.total_steps=40

## What this proves — and what's next

✅ Kaggle can build the environment, the tests pass, the trainer runs, and **resume works
across separate invocations with no discontinuity**.

**Next (Phase 1b):** a `kaggle_phase1_sft.ipynb` notebook that
1. attaches a pre-staged **Kaggle Dataset** with GPT-2 + GSM8k-Aug/eval sets and sets
   `HF_HUB_OFFLINE=1` (Internet can then be turned **Off**),
2. runs the **No-CoT-SFT / CoT-SFT** baselines on GPU, and
3. persists checkpoints to `/kaggle/working` so a committed version can be re-attached and
   resumed past the session cap.